# Black-box & derivative-free optimization — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def zero_one(pred,y): return np.mean(np.asarray(pred)!=np.asarray(y))
print('setup ok')

## Search when gradients are unavailable, unreliable, or too expensive
Derivative-free optimization survives when the function is a black box. These five examples show finite-difference probing, coordinate pattern search, Nelder-Mead simplex movement, random search improving with budget, and robustness to noisy objective values.

In [ ]:
# 1) finite difference estimates a slope from function calls
f=lambda x:(x-2)**2; x=0.; h=0.1; fd=(f(x+h)-f(x-h))/(2*h); true=2*(x-2)
xx=np.linspace(-1,3,200); plt.figure(figsize=(5,3)); plt.plot(xx,f(xx)); plt.scatter([x-h,x+h],[f(x-h),f(x+h)],c='r'); plt.title(f'finite-difference slope {fd:.1f}')
assert abs(fd-true)<1e-12

In [ ]:
# 2) coordinate pattern search accepts improving probes
f2=lambda z:(z[0]-1)**2+(z[1]+2)**2; x=np.array([0.,0.]); step=1.; path=[x.copy()]
for d in [np.array([1.,0.]),np.array([0.,-1.]),np.array([0.,-1.])]:
    if f2(x+step*d)<f2(x): x=x+step*d; path.append(x.copy())
P=np.array(path); plt.figure(figsize=(5,4)); plt.plot(P[:,0],P[:,1],'-o'); plt.scatter([1],[-2],c='r'); plt.title('pattern search path')
assert np.allclose(x,[1,-2])

In [ ]:
# 3) simplex reflection improves the worst vertex
verts=np.array([[0.,0.],[2.,0.],[0.,2.]]); vals=np.array([f2(v) for v in verts]); worst=np.argmax(vals); centroid=verts[np.arange(3)!=worst].mean(axis=0); reflected=centroid+(centroid-verts[worst])
plt.figure(figsize=(5,4)); plt.fill(verts[:,0],verts[:,1],alpha=.2); plt.scatter([reflected[0]],[reflected[1]],c='r'); plt.title('Nelder-Mead reflection')
assert f2(reflected)<vals[worst]

In [ ]:
# 4) random search improves with more trials
rng=np.random.default_rng(0); budgets=[5,20,100,500]; best=[]
for B in budgets:
    pts=rng.uniform(-3,3,size=(B,2)); best.append(min(f2(p) for p in pts))
plt.figure(figsize=(5,3)); plt.loglog(budgets,best,'-o'); plt.title('more budget finds better points')
assert best[-1]<best[0]

In [ ]:
# 5) averaging repeats combats noisy objectives
rng=np.random.default_rng(1); candidates=np.array([0.,1.,2.,3.]); noisy=lambda x,n: np.mean((x-2)**2+rng.normal(0,0.5,n))
one=np.array([noisy(x,1) for x in candidates]); many=np.array([noisy(x,50) for x in candidates])
plt.figure(figsize=(5,3)); plt.plot(candidates,one,'o-',label='1 eval'); plt.plot(candidates,many,'s-',label='50 eval avg'); plt.legend(); plt.title('replication stabilizes black-box search')
assert candidates[np.argmin(many)]==2